# 06 · Inference: the risk engine behind the API

The deployed service ships only `artifacts/model_bundle.joblib` and `artifacts/demo_state.joblib`
(the feature store replayed to the end of the window + the last graph snapshot): a few MB, so it fits
Render's 512 MB free tier. Start it with `uvicorn src.serve.api:app`.

In [1]:
# --- setup: find the repo root (works locally, in Colab and on Kaggle) -------------------------
import os, sys, shutil, subprocess
from pathlib import Path

REPO_URL = ""          # optional: your GitHub repo URL, cloned if the code is not found
ROOT = None
for cand in [Path.cwd(), Path.cwd().parent, *Path("/kaggle/input").glob("*")] if Path("/kaggle/input").exists() else [Path.cwd(), Path.cwd().parent]:
    if (cand / "src" / "pipeline.py").exists():
        ROOT = cand
        break
if ROOT is None and REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "prohori"], check=True)
    ROOT = Path("prohori").resolve()
assert ROOT is not None, "Put this notebook next to the repo (or add the repo as a Kaggle dataset / set REPO_URL)"
if str(ROOT).startswith("/kaggle/input"):          # Kaggle inputs are read-only: work on a copy
    dst = Path("/kaggle/working/prohori")
    shutil.copytree(ROOT, dst, dirs_exist_ok=True)
    ROOT = dst
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

repo root: H:\Tacos Projects\Hackathon\Asking for a Friend


In [2]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40, "display.width", 200)
from src.common.config import load_config, resolve
from src.models import plots
plots.style()
cfg = load_config()                 # config.yaml (seed 42, full scale)
REPORTS = resolve(cfg, "reports_dir")

In [3]:
from src.serve.scorer import RiskEngine
eng = RiskEngine()
wallets = list(eng.store.w)[:3]
r = eng.score(dict(sender_id=wallets[0], receiver_id=wallets[1], amount=1500))
print(r["band"], r["risk_score"], [x["en"] for x in r["reasons"]])

ALLOW 0.0 ['You have never sent money to this number before', 'Unusual time for this customer (00:00)']


## A large first-time transfer from an unknown phone at 02:00

In [4]:
t = eng.now + 3600 * ((26 - (eng.now % 86400) // 3600) % 24)       # next 02:00
r = eng.score(dict(sender_id=wallets[0], receiver_id=wallets[2], amount=25000, device_id="DV9999999", ts_sec=int(t)))
print(r["band"], r["risk_score"])
for x in r["reasons"]:
    print("-", x["en"], "|", x["bn"])
print(r["customer_message"]["bn"])

ALLOW 0.3
- Tk 25,000 is 1.0x the largest amount this customer has sent | ৳২৫,০০০ আপনার আগের সর্বোচ্চ লেনদেনের ১.০ গুণ
- Unusual time for this customer (02:00) | আপনার জন্য অস্বাভাবিক সময় (০২:০০)
- You have never sent money to this number before | এই নম্বরে আপনি আগে কখনো টাকা পাঠাননি
- Very unusual for this customer's own past behaviour (top 2% anomaly) | এই গ্রাহকের নিজের আগের আচরণের তুলনায় অত্যন্ত অস্বাভাবিক



## Recorded outcomes for the planted scenarios

In [5]:
pd.DataFrame(eng.state["demo"])[["id", "title", "expected", "actual", "risk_score", "pass"]]

,id,title,expected,actual,risk_score,pass
0,SC-01,SIM-swap takeover of a business owner at 01:40,HOLD,HOLD,100.0,True
1,SC-02,Agent-register harvest: OTP takeover drains 12...,STEP_UP,HOLD,100.0,True
2,SC-03,"Collector wallet: 2 days old, 14 strangers pai...",STEP_UP,HOLD,100.0,True
3,SC-04,Mule chain through a 400-day-old dormant accou...,STEP_UP,HOLD,98.0,True
4,SC-05,'Jail guard' pressure scam: 60+ USSD farmer ca...,NUDGE,NUDGE,30.5,True
5,SC-06,"Rogue agent: night cash-outs for the ring, ~8-...",FLAGGED,FLAGGED,85.8,True
6,SC-07,"Stolen card: 2 x 25,000 Add Money at 03:10, ca...",HOLD,HOLD,100.0,True
7,SC-08,Fake phone seller: 9 advance payments in 3 day...,NUDGE,HOLD,88.6,True
8,SB-01,Busy shop owner's personal wallet: 12 payers i...,ALLOW,ALLOW,10.6,True
9,SB-02,"Legit new phone, then usual bill + 2,000 to mo...",ALLOW,ALLOW,15.3,True
